# KFold 앙상블



# Seed Ensemble ( 시드 앙상블 )

- 시드 앙상블은 여러 시드를 활용해 모델을 학습하고 그 결과를 합쳐 최종 예측을 만드는 앙상블 기법
- 시드(seed)는 무작위성을 제어하기 위한 값, 동일한 시드를 설정하면 랜덤 요소가 있더라도 동일한 결과를 얻을 수 있다.

```python
import numpy as np
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

all_predictions = []
test_predictions = []
seeds = [42, 77, 2024]

# 시드 앙상블 사용
for seed in seeds:
    seed_et_model = ExtraTreesRegressor(n_estimators = 20, random_state = seed)
    seed_et_model.fit(x_train, y_train)
    seed_et_pred = seed_et_model.predict(x_valid)
    all_predictions.append(seed_et_pred)
    
seed_pred = np.mean(all_predictions, axis = 0)
rmse_seed_val = root_mean_squared_error(y_valid, seed_pred)

# 단일 모델 사용
et_model = ExtraTreesRegressor(n_estimators = 20, random_state = 42)
et_model.fit(x_train, y_train)
et_pred = et_model.predict(x_valid)

rmse_one_val = root_mean_squared_error(y_valid, et_pred)

print(f"시드 앙상블 모델의 RMSE 검증점수 : {rmse_seed_val}")
print(f"ExtraTrees 단일 모델의 RMSE 검증점수 : {rmse_one_val}")
```

---

### Result 

```
시드 앙상블 모델의 RMSE 검증점수 : 442.23487708223166
ExtraTrees 단일 모델의 RMSE 검증점수 : 458.474566984286
```

## KFold 교차 검증을 이용한 모델 학습 및 평가 

- [KFold](./8.1.1.Ensemble.ipynb)  

```python
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import ExtraTreesRegressor

# KFold 설정
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 모델과 결과를 저장할 리스트
models = []
scores = []

# KFold를 사용한 앙상블 훈련
# kf split을 사용하여 drop_train데이터를 학습용 인덱스(train_idx)와 검증용 인덱스(valid_idx)로 나눈다. 
for train_idx, valid_idx in kf.split(drop_train):
    x_train, x_valid = drop_train.iloc[train_idx], drop_train.iloc[valid_idx]
    y_train, y_valid = target.iloc[train_idx], target.iloc[valid_idx]

    # 모델 초기화 및 학습
    model = ExtraTreesRegressor(n_estimators=20, random_state=42)
    model.fit(x_train, y_train)

    # 예측 및 점수 획득
    y_pred = model.predict(x_valid)
    score = root_mean_squared_error(y_valid, y_pred)
    print(score)

    # 모델 및 점수 저장
    models.append(model)
    scores.append(score)

# 최종 평균 정확도
print(f"KFold 앙상블의 평균 RMSE 검증점수: {np.mean(scores)}")

```

---

```
405.40765514211256
434.24464340517426
332.1895003314144
316.64179202455534
394.3753151561916
KFold 앙상블의 평균 RMSE 검증점수: 376.57178121188963
```

```python
# 각 모델에 다른 데이터를 이용하여 학습
[f"{i}번째 모델의 피처중요도{models[i].feature_importances_}" for i in range(5)]
```

- `feature_importances_`가 높을 수록 좋은것 

---

```
['0번째 모델의 피처중요도[0.08732231 0.14450925 0.66021327 0.05415564 0.05379954]',
 '1번째 모델의 피처중요도[0.07268239 0.13889763 0.67936706 0.05200196 0.05705096]',
 '2번째 모델의 피처중요도[0.06943726 0.16200811 0.6593977  0.05627491 0.05288202]',
 '3번째 모델의 피처중요도[0.102822   0.14524309 0.58874819 0.10377085 0.05941588]',
 '4번째 모델의 피처중요도[0.092089   0.1687682  0.63945088 0.05645424 0.04323769]']
```

## KFold 평균 앙상블 적용 

```python
def ensemble_predict(new_x_data):
    all_predictions = np.array([model.predict(new_x_data) for model in models])
    final_predictions = np.mean(all_predictions, axis=0).astype(int)
    return final_predictions

# 새로운 데이터에 대한 예측
X_new = drop_test[0:5]
y_new_pred = ensemble_predict(X_new)

print(f"새로운 데이터에 대한 앙상블 예측: {y_new_pred}")
```

---

```
새로운 데이터에 대한 앙상블 예측: [4224 5572 3896 3174 3499]
```

## KFold 가중 평균 앙상블 적용 

- 가중 평균 앙상블 예측을 수행하는 방법

```python
def weighted_ensemble_predict(new_x_data):
    
    # 가중치 설정 및 정규화
    weights = [1.0 / score for score in scores]
    weights = np.array(weights) / np.sum(weights)

    # 각 모델의 예측을 가중치에 따라 조정
    all_predictions = np.array([model.predict(new_x_data) for model in models])
    weighted_predictions = np.dot(weights, all_predictions)

    # 최종 예측 산출
    final_predictions = weighted_predictions.astype(int)
    return final_predictions

# 새로운 데이터에 대한 예측
X_new = drop_test[0:5]
y_new_pred = weighted_ensemble_predict(X_new)

print(f"새로운 데이터에 대한 앙상블 예측: {y_new_pred}")
```

---

- Result 

```
새로운 데이터에 대한 앙상블 예측: [4226 5568 3896 3171 3502]
```


- [np dot explain](./tip/npdot.ipynb). 

```
all_predictions

array([[4145.  , 5570.  , 3841.25, 3526.25, 3456.25],
       [4220.  , 5695.  , 3801.25, 2923.75, 3452.5 ],
       [4330.  , 5657.5 , 4135.  , 3078.75, 3582.5 ],
       [4173.75, 5480.  , 3665.  , 3126.25, 3475.  ],
       [4255.  , 5457.5 , 4040.  , 3216.25, 3530.  ]])

weights

array([0.18307263, 0.17091528, 0.22342381, 0.23439434, 0.18819394])
```

# 위 함수를 풀어서 작성 

```python
# 위 함수를 풀어서 작성한 경우
sample_pred = np.array([model.predict(drop_test[0:2]) for model in models])
sample_weight = [1.0 / score for score in scores]
sample_weight = np.array(sample_weight) / np.sum(sample_weight)

print(f"모델의 예측값은 다음과 같습니다. \n{sample_pred}")
print(f"각 모델의 가중치는 다음과 같습니다. \n {sample_weight}")

np.dot(sample_weight, sample_pred).round().astype(int)
```

---

- Result 

```
모델의 예측값은 다음과 같습니다. 
[[4145.   5570.  ]
 [4220.   5695.  ]
 [4330.   5657.5 ]
 [4173.75 5480.  ]
 [4255.   5457.5 ]]
각 모델의 가중치는 다음과 같습니다. 
 [0.18307263 0.17091528 0.22342381 0.23439434 0.18819394]
array([4227, 5569])
```

- dot을 하는 이유 : 각 모델별 예측값에 대해서 가중치를 곱해서(가중치의 합은 1 ) 평균을 구하는 식이 됨. 

# 시드 기반 KFold 앙상블 

```python
# 시드 설정
seeds = [42, 77, 2024]

# 결과를 저장할 배열 초기화
all_scores = []
all_models = [] 

# 모든 시드에 대해 모델 초기화 및 훈련
for seed in seeds:
    fold_scores = []

    # 각 시드에 대해 KFold 설정
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)

    # KFold를 사용한 앙상블 훈련
    for train_idx, valid_idx in kf.split(drop_train):
        x_train, x_valid = drop_train.iloc[train_idx], drop_train.iloc[valid_idx]
        y_train, y_valid = target.iloc[train_idx], target.iloc[valid_idx]

        # 모델 초기화 및 학습
        model = ExtraTreesRegressor(n_estimators=20, random_state=seed)
        model.fit(x_train, y_train)
        y_pred = model.predict(x_valid)
        
        score = root_mean_squared_error(y_valid, y_pred)
        all_models.append(model) 
        fold_scores.append(score)
        print(f"Seed {seed} RMSE: {score}")
    
    # 각 시드 별 평균 점수 계산 및 저장
    average_seed_score = np.mean(fold_scores)
    all_scores.append(average_seed_score)
    print(f"Seed {seed} RMSE 평균 -> {average_seed_score}")
    print("-"*40)

# 모든 시드 및 폴드에 대한 최종 평균 정확도
overall_average_score = np.mean(all_scores)
print(f"전체 평균 RMSE 검증점수: {overall_average_score}")
```

===

## Result 

```
Seed 42 RMSE: 405.40765514211256
Seed 42 RMSE: 434.24464340517426
Seed 42 RMSE: 332.1895003314144
Seed 42 RMSE: 316.64179202455534
Seed 42 RMSE: 394.3753151561916
Seed 42 RMSE 평균 -> 376.57178121188963
----------------------------------------
Seed 77 RMSE: 382.6344250859684
Seed 77 RMSE: 307.5259577237555
Seed 77 RMSE: 347.61385763766543
Seed 77 RMSE: 380.4177480658313
Seed 77 RMSE: 435.40570161632013
Seed 77 RMSE 평균 -> 370.71953802590815
----------------------------------------
Seed 2024 RMSE: 245.75485421363547
Seed 2024 RMSE: 418.83013509681626
Seed 2024 RMSE: 386.9036434925705
Seed 2024 RMSE: 393.3016771374493
Seed 2024 RMSE: 459.3996205523426
Seed 2024 RMSE 평균 -> 380.83798609856285
----------------------------------------
전체 평균 RMSE 검증점수: 376.0431017787869
```

## 학습된 데이터 모델을 활용하여 새로운 데이터에 대한 예측을 수행 

```python
import numpy as np

# KFold 앙상블을 사용하여 새로운 데이터에 대한 예측을 수행하는 함수
def kfold_seed_mean_ensemble(models_list, new_data):
    # 각 모델의 예측을 수행하고 결과를 리스트에 저장
    all_predictions = np.array([model.predict(new_data) for model in models])
    
    # 모든 모델의 예측 결과의 평균을 계산
    final_predictions = np.mean(all_predictions, axis=0)
    
    return final_predictions

# 새로운 데이터 drop_test에 대한 예측 수행
y_new_pred = kfold_seed_mean_ensemble(models, drop_test)
submission['Body Mass (g)'] = y_new_pred
submission.head()
```

===

## Result 

```

id	Body Mass (g)
0	0	4224.75
1	1	5572.00
2	2	3896.50
3	3	3174.25
4	4	3499.25

```